# Water Quality Analysis — Trebujena, Cadiz (Guadalquivir)

Analisis de calidad de agua usando **Sentinel-2 L2A** via **Google Earth Engine** para un territorio de 6 ha en las marismas del Guadalquivir.

### Indicadores:
1. **Water Thresholding** — Deteccion de masas de agua (MNDWI/NDWI/SWI)
2. **Water Turbidity** — Turbidez en NTU (modelo empirico Se2WaQ)
3. **Water Chlorophyll** — Clorofila-a via NDCI + modelo empirico Chl_a

**Autor:** WildSquare | **Licencia:** CC BY-SA 4.0

## 1. Autenticacion y Setup

In [ ]:
import ee
import json
import folium
from IPython.display import display, Image, HTML
from datetime import datetime

# Autenticar con Google Earth Engine
ee.Authenticate()
ee.Initialize(project='earthengine-legacy')
print('Google Earth Engine inicializado correctamente')

## 2. Configuracion del Territorio

In [ ]:
# ===== CONFIGURACION =====
# Puedes cambiar estas coordenadas para analizar otro territorio

TERRITORY_NAME = 'Guadalquivir - Trebujena, Cadiz'

# Poligono del territorio (~6 ha en marismas del Guadalquivir)
TERRITORY_COORDS = [
    [-6.1780, 36.8680],
    [-6.1750, 36.8680],
    [-6.1750, 36.8650],
    [-6.1720, 36.8650],
    [-6.1720, 36.8620],
    [-6.1780, 36.8620],
    [-6.1780, 36.8680],
]

# Periodo de analisis
DATE_START = '2024-06-01'
DATE_END   = '2025-03-01'

# Parametros Sentinel-2
CLOUD_MAX_PCT = 15   # Max % nubes
SCALE = 10           # Resolucion en metros

# Umbrales de deteccion de agua
MNDWI_THR = 0.1
NDWI_THR  = 0.2
SWI_THR   = 0.03

# Crear geometria
geom = ee.Geometry.Polygon([TERRITORY_COORDS])
center = geom.centroid().getInfo()['coordinates']
print(f'Territorio: {TERRITORY_NAME}')
print(f'Centro: {center[1]:.4f}N, {center[0]:.4f}W')

## 3. Obtener imagen Sentinel-2

In [ ]:
# Buscar la mejor imagen Sentinel-2 L2A
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(geom)
    .filterDate(DATE_START, DATE_END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_MAX_PCT))
    .sort('CLOUDY_PIXEL_PERCENTAGE')
)

count = collection.size().getInfo()
print(f'Imagenes encontradas con <{CLOUD_MAX_PCT}% nubes: {count}')

if count == 0:
    print('ERROR: No se encontraron imagenes. Aumenta CLOUD_MAX_PCT o el rango de fechas.')
else:
    image = collection.first()
    image_date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    image_id = image.get('system:index').getInfo()
    cloud_pct = image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    print(f'Imagen seleccionada: {image_id}')
    print(f'Fecha: {image_date} | Nubes: {cloud_pct:.1f}%')

## 4. Calcular indices de calidad de agua

In [ ]:
def add_water_indices(img):
    """Anade todos los indices de calidad de agua a la imagen."""
    ndwi  = img.normalizedDifference(['B3', 'B8']).rename('NDWI')
    mndwi = img.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    swi   = img.normalizedDifference(['B5', 'B11']).rename('SWI')
    ndci  = img.normalizedDifference(['B5', 'B4']).rename('NDCI')

    ratio = img.select('B3').divide(img.select('B1'))
    turbidity = ratio.multiply(8.93).subtract(6.39).rename('TURBIDITY')
    chla = ratio.pow(3.94).multiply(4.26).rename('CHL_A')

    return img.addBands([ndwi, mndwi, swi, ndci, turbidity, chla])


def get_water_mask(img):
    """Mascara de agua: (MNDWI>0.1) OR (NDWI>0.2) OR (SWI>0.03) OR SCL==6"""
    return (
        img.select('MNDWI').gt(MNDWI_THR)
        .Or(img.select('NDWI').gt(NDWI_THR))
        .Or(img.select('SWI').gt(SWI_THR))
        .Or(img.select('SCL').eq(6))
        .rename('WATER')
    )


# Calcular indices
image_idx = add_water_indices(image)
water_mask = get_water_mask(image_idx)
cloud_mask = image.select('SCL').eq(8).Or(image.select('SCL').eq(9)).rename('CLOUD')
image_all = image_idx.addBands([water_mask, cloud_mask])

# Imagen solo agua (para estadisticas de turbidez y clorofila)
water_only = image_all.updateMask(water_mask)

print('Indices calculados: NDWI, MNDWI, SWI, NDCI, TURBIDITY, CHL_A')

## 5. Resultados: Water Thresholding

In [ ]:
# === WATER THRESHOLDING ===
print('=' * 60)
print('  INDICADOR 1: WATER THRESHOLDING')
print('  Deteccion de masas de agua (MNDWI/NDWI/SWI)')
print('=' * 60)

# Estadisticas de los indices
stats = image_all.select(['NDWI', 'MNDWI', 'SWI', 'WATER', 'CLOUD']).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=geom,
    scale=SCALE,
    maxPixels=1e7,
    bestEffort=True,
).getInfo()

# Area de agua
water_area = water_mask.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=geom, scale=SCALE, maxPixels=1e7, bestEffort=True
).get('WATER').getInfo()

total_area = ee.Image.pixelArea().reduceRegion(
    reducer=ee.Reducer.sum(), geometry=geom, scale=SCALE, maxPixels=1e7, bestEffort=True
).get('area').getInfo()

water_ha = (water_area or 0) / 10000
total_ha = (total_area or 0) / 10000
water_pct = (water_ha / total_ha * 100) if total_ha > 0 else 0

print(f'\n  Area total:         {total_ha:.2f} ha')
print(f'  Superficie de agua: {water_ha:.2f} ha ({water_pct:.1f}%)')
print(f'  Fraccion de nubes:  {(stats.get("CLOUD", 0) or 0) * 100:.1f}%')
print(f'\n  Umbrales utilizados:')
print(f'    MNDWI > {MNDWI_THR}  |  NDWI > {NDWI_THR}  |  SWI > {SWI_THR}')
print(f'\n  Valores medios en territorio:')
print(f'    MNDWI: {stats.get("MNDWI", "N/A")}')
print(f'    NDWI:  {stats.get("NDWI", "N/A")}')
print(f'    SWI:   {stats.get("SWI", "N/A")}')

In [ ]:
# Visualizar mascara de agua
geom_info = geom.getInfo()

url_water = water_mask.getThumbUrl({
    'region': geom_info, 'dimensions': '600x600', 'format': 'png',
    'min': 0, 'max': 1, 'palette': ['FFFFFF', '0033CC'],
})
url_rgb = image.select(['B4', 'B3', 'B2']).getThumbUrl({
    'region': geom_info, 'dimensions': '600x600', 'format': 'png',
    'min': 0, 'max': 3000,
})

display(HTML(f'''
<div style="display:flex; gap:20px;">
  <div style="text-align:center">
    <h4>Color verdadero</h4>
    <img src="{url_rgb}" width="400">
  </div>
  <div style="text-align:center">
    <h4>Mascara de agua (azul = agua)</h4>
    <img src="{url_water}" width="400">
  </div>
</div>
'''))

## 6. Resultados: Water Turbidity

In [ ]:
# === WATER TURBIDITY ===
print('=' * 60)
print('  INDICADOR 2: WATER TURBIDITY')
print('  Turbidez (NTU) — Modelo Se2WaQ')
print('  Formula: Turb = 8.93 x (B03/B01) - 6.39')
print('=' * 60)

turb_stats = water_only.select('TURBIDITY').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=geom, scale=SCALE, maxPixels=1e7, bestEffort=True,
).getInfo()

turb_mean = turb_stats.get('TURBIDITY_mean')
turb_min  = turb_stats.get('TURBIDITY_min')
turb_max  = turb_stats.get('TURBIDITY_max')

def classify_turb(ntu):
    if ntu is None: return 'Sin datos'
    if ntu < 4: return 'Agua limpia'
    elif ntu < 8: return 'Turbidez baja'
    elif ntu < 12: return 'Turbidez moderada'
    elif ntu < 16: return 'Turbidez alta'
    elif ntu < 20: return 'Turbidez muy alta'
    else: return 'Turbidez extrema'

print(f'\n  Media (pixeles agua): {turb_mean:.2f} NTU' if turb_mean else '\n  Media: Sin datos')
print(f'  Minimo:               {turb_min:.2f} NTU' if turb_min else '  Minimo: Sin datos')
print(f'  Maximo:               {turb_max:.2f} NTU' if turb_max else '  Maximo: Sin datos')
print(f'  Clasificacion:        {classify_turb(turb_mean)}')
print(f'\n  Escala: 0=limpio, 4=baja, 8=moderada, 12=alta, 16=muy alta, 20=extrema')

In [ ]:
# Visualizar turbidez
url_turb = water_only.select('TURBIDITY').getThumbUrl({
    'region': geom_info, 'dimensions': '600x600', 'format': 'png',
    'min': 0, 'max': 20,
    'palette': ['4973F2', '82D35F', 'FEFD05', 'FD0004', '8E2026', 'D97CF5'],
})

display(HTML(f'''
<div style="text-align:center">
  <h4>Mapa de Turbidez (NTU) — Solo pixeles de agua</h4>
  <img src="{url_turb}" width="500">
  <p style="color:gray">Azul=limpio | Verde=baja | Amarillo=moderada | Rojo=alta | Purpura=extrema</p>
</div>
'''))

## 7. Resultados: Water Chlorophyll

In [ ]:
# === WATER CHLOROPHYLL ===
print('=' * 60)
print('  INDICADOR 3: WATER CHLOROPHYLL')
print('  NDCI + Clorofila-a empirica')
print('=' * 60)

# NDCI stats
ndci_stats = water_only.select('NDCI').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=geom, scale=SCALE, maxPixels=1e7, bestEffort=True,
).getInfo()

# Chl-a stats
chla_stats = water_only.select('CHL_A').reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), sharedInputs=True),
    geometry=geom, scale=SCALE, maxPixels=1e7, bestEffort=True,
).getInfo()

ndci_mean = ndci_stats.get('NDCI_mean')
chla_mean = chla_stats.get('CHL_A_mean')

def classify_ndci(v):
    if v is None: return 'Sin datos'
    if v < -0.2: return 'Clorofila muy baja (agua clara)'
    elif v < 0.0: return 'Clorofila baja'
    elif v < 0.2: return 'Clorofila moderada'
    elif v < 0.4: return 'Clorofila elevada'
    else: return 'Clorofila alta / bloom algal'

def classify_trophic(v):
    if v is None: return 'Sin datos'
    if v < 6: return 'Oligotrofico'
    elif v < 12: return 'Mesotrofico'
    elif v < 20: return 'Meso-eutrofico'
    elif v < 30: return 'Eutrofico'
    elif v < 50: return 'Hipereutrofico'
    else: return 'Bloom algal'

print(f'\n  --- NDCI (Normalized Difference Chlorophyll Index) ---')
print(f'  Formula: NDCI = (B05 - B04) / (B05 + B04)')
print(f'  Media (agua):  {ndci_mean:.4f}' if ndci_mean else '  Media: Sin datos')
print(f'  Rango:         [{ndci_stats.get("NDCI_min", "?"):.4f}, {ndci_stats.get("NDCI_max", "?"):.4f}]' if ndci_stats.get('NDCI_min') else '')
print(f'  Clasificacion: {classify_ndci(ndci_mean)}')

print(f'\n  --- Clorofila-a (modelo empirico Se2WaQ) ---')
print(f'  Formula: Chl_a = 4.26 x (B03/B01)^3.94  [mg/m3]')
print(f'  Media (agua):  {chla_mean:.2f} mg/m3' if chla_mean else '  Media: Sin datos')
print(f'  Rango:         [{chla_stats.get("CHL_A_min", "?"):.2f}, {chla_stats.get("CHL_A_max", "?"):.2f}] mg/m3' if chla_stats.get('CHL_A_min') else '')
print(f'  Estado trofico: {classify_trophic(chla_mean)}')

In [ ]:
# Visualizar NDCI y Chl-a
url_ndci = water_only.select('NDCI').getThumbUrl({
    'region': geom_info, 'dimensions': '600x600', 'format': 'png',
    'min': -0.2, 'max': 0.4,
    'palette': ['313695', '4575b4', 'e0f3f8', 'fee090', 'fdae61', 'f46d43', 'a50026'],
})

url_chla = water_only.select('CHL_A').getThumbUrl({
    'region': geom_info, 'dimensions': '600x600', 'format': 'png',
    'min': 0, 'max': 50,
    'palette': ['4973F2', '82D35F', 'FEFD05', 'FD0004', '8E2026', 'D97CF5'],
})

display(HTML(f'''
<div style="display:flex; gap:20px;">
  <div style="text-align:center">
    <h4>NDCI (Indice de Clorofila)</h4>
    <img src="{url_ndci}" width="400">
    <p style="color:gray">Azul=baja | Amarillo=moderada | Rojo=alta</p>
  </div>
  <div style="text-align:center">
    <h4>Clorofila-a (mg/m3)</h4>
    <img src="{url_chla}" width="400">
    <p style="color:gray">Azul=oligotrofico | Verde=mesotrofico | Rojo=eutrofico</p>
  </div>
</div>
'''))

## 8. Mapa interactivo

In [ ]:
# Mapa interactivo con Folium
m = folium.Map(location=[center[1], center[0]], zoom_start=15)

# Capa: color verdadero
rgb_tiles = image.select(['B4', 'B3', 'B2']).getMapId({'min': 0, 'max': 3000})
folium.TileLayer(
    tiles=rgb_tiles['tile_fetcher'].url_format,
    attr='Sentinel-2', name='Color verdadero', overlay=True
).add_to(m)

# Capa: mascara de agua
water_tiles = water_mask.getMapId({'min': 0, 'max': 1, 'palette': ['00000000', '0033CCAA']})
folium.TileLayer(
    tiles=water_tiles['tile_fetcher'].url_format,
    attr='Water mask', name='Agua detectada', overlay=True
).add_to(m)

# Capa: turbidez
turb_tiles = water_only.select('TURBIDITY').getMapId({
    'min': 0, 'max': 20,
    'palette': ['4973F2', '82D35F', 'FEFD05', 'FD0004', '8E2026', 'D97CF5']
})
folium.TileLayer(
    tiles=turb_tiles['tile_fetcher'].url_format,
    attr='Turbidity', name='Turbidez (NTU)', overlay=True
).add_to(m)

# Capa: NDCI
ndci_tiles = water_only.select('NDCI').getMapId({
    'min': -0.2, 'max': 0.4,
    'palette': ['313695', '4575b4', 'e0f3f8', 'fee090', 'fdae61', 'f46d43', 'a50026']
})
folium.TileLayer(
    tiles=ndci_tiles['tile_fetcher'].url_format,
    attr='NDCI', name='Clorofila (NDCI)', overlay=True
).add_to(m)

# Contorno del territorio
folium.GeoJson(
    geom.getInfo(),
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'red', 'weight': 2}
).add_to(m)

folium.LayerControl().add_to(m)
m

## 9. Resumen y exportar JSON

In [ ]:
# Resumen final
results = {
    'territorio': TERRITORY_NAME,
    'fecha_analisis': datetime.now().isoformat(),
    'imagen': {'id': image_id, 'fecha': image_date, 'nubes_pct': round(cloud_pct, 1)},
    'area': {
        'total_ha': round(total_ha, 2),
        'agua_ha': round(water_ha, 2),
        'agua_pct': round(water_pct, 1),
    },
    'indicadores': {
        'water_thresholding': {
            'MNDWI_medio': round(stats.get('MNDWI', 0) or 0, 4),
            'NDWI_medio': round(stats.get('NDWI', 0) or 0, 4),
            'SWI_medio': round(stats.get('SWI', 0) or 0, 4),
        },
        'turbidez_NTU': {
            'media': round(turb_mean, 2) if turb_mean else None,
            'min': round(turb_min, 2) if turb_min else None,
            'max': round(turb_max, 2) if turb_max else None,
            'clasificacion': classify_turb(turb_mean),
        },
        'clorofila': {
            'NDCI_medio': round(ndci_mean, 4) if ndci_mean else None,
            'NDCI_clasificacion': classify_ndci(ndci_mean),
            'Chla_media_mg_m3': round(chla_mean, 2) if chla_mean else None,
            'estado_trofico': classify_trophic(chla_mean),
        },
    },
}

# Guardar JSON
with open('water_quality_results.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(json.dumps(results, indent=2, ensure_ascii=False))
print('\nResultados guardados en water_quality_results.json')

## Referencias

1. McFeeters, S.K. (1996). *The Use of the NDWI in the Delineation of Open Water Features.* Int. J. Remote Sensing, 17(7), 1425-1432.
2. Xu, H. (2006). *Modification of NDWI to Enhance Open Water Features.* Int. J. Remote Sensing, 27(14), 3025-3033.
3. Mishra, S. & Mishra, D.R. (2012). *Normalized Difference Chlorophyll Index.* Remote Sens. Environ., 117, 394-406.
4. Potes, M. et al. (2018). *Use of Sentinel 2-MSI for Water Quality Monitoring at Alqueva Reservoir.* Proc. IAHS, 380, 73-79.
5. Toming, K. et al. (2016). *First Experiences in Mapping Lake Water Quality with Sentinel-2 MSI.* Remote Sensing, 8(8), 640.